In [1]:
"""

Refined predictor-corrector pipeline for periodic variable-speed linear advection.

PDE
---
    u_t + a(x;beta) u_x = 0,   x in [0,1], t in [0,1]
    a(x;beta) = 1 + beta * sin(2*pi*x),   |beta| < 1

BC
--
    u(0,t) = u(1,t)

IC
--
    u(x,0) = u0(x; x0, nu)
where u0 is a periodic Gaussian.

Task parameters
---------------
    p = (x0, nu, beta)

Pipeline
--------
1) Train/load a generic dynamic-basis meta-SPINN predictor over the task family.
   The predictor is allowed to be diffusive. Its role is only to provide:
      - dynamic center locations
      - dynamic widths
      - a coarse predicted field u_pred(x,t)

2) For each test task p:
   - discard predictor field values as the final answer,
   - but use predictor geometry/value/gradient to build a frozen PIELM hidden layer.

3) Refined frozen hidden layer:
   (A) More FIXED background centers on a denser spacetime grid.
       Their widths are tied to grid spacing.
   (B) Dynamic centers guided by predictor internal centers/widths.
   (C) Additional narrow centers placed in regions of HIGH PREDICTOR GRADIENT.

4) Solve the PIELM outer coefficients analytically by ridge-regularized LS:
       u(x,t;p) = u0(x;p) + t * sum_j c_j * phi_j(x,t)

"""

import os
import csv
import math
import random
import argparse
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim

# ============================================================
# 1) Reproducibility
# ============================================================

SEED = 1234
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# ============================================================
# 2) Utilities
# ============================================================

def periodic_distance_torch(x, y):
    return torch.remainder(x - y + 0.5, 1.0) - 0.5

def periodic_distance_np(x, y):
    return np.remainder(x - y + 0.5, 1.0) - 0.5

def periodic_gaussian(z, mu, nu):
    if isinstance(z, torch.Tensor):
        d = periodic_distance_torch(z, mu)
        return torch.exp(-d**2 / (2.0 * nu**2))
    d = periodic_distance_np(z, mu)
    return np.exp(-d**2 / (2.0 * nu**2))

def periodic_gaussian_np(z, mu, nu):
    d = periodic_distance_np(z, mu)
    return np.exp(-d**2 / (2.0 * nu**2))

def periodic_gaussian_x_derivative_torch(x, mu, nu):
    d = periodic_distance_torch(x, mu)
    g = torch.exp(-d**2 / (2.0 * nu**2))
    return -(d / (nu**2)) * g

def a_of_x_torch(x, beta):
    return 1.0 + beta * torch.sin(2.0 * math.pi * x)

def a_of_x_np(x, beta):
    return 1.0 + beta * np.sin(2.0 * math.pi * x)

def fourier_time_features(t, n_freq=4):
    feats = [t]
    for k in range(1, n_freq + 1):
        feats.append(torch.sin(2.0 * math.pi * k * t))
        feats.append(torch.cos(2.0 * math.pi * k * t))
    return torch.cat(feats, dim=-1)

# ============================================================
# 3) Exact solution for evaluation only
# ============================================================

class TravelTimeMap:
    def __init__(self, beta, n_grid=20001):
        self.beta = float(beta)
        self.n_grid = int(n_grid)

        x = np.linspace(0.0, 1.0, self.n_grid)
        a = a_of_x_np(x, self.beta)
        inv_a = 1.0 / a

        F = np.zeros_like(x)
        dx = x[1] - x[0]
        F[1:] = np.cumsum(0.5 * (inv_a[:-1] + inv_a[1:]) * dx)

        self.x = x
        self.F = F
        self.P = F[-1]

    def forward_F(self, xq):
        return np.interp(np.mod(xq, 1.0), self.x, self.F)

    def inverse_F(self, tauq):
        return np.interp(np.mod(tauq, self.P), self.F, self.x)

def exact_solution_variable_speed(X, T, x0, nu, beta, n_grid_map=20001):
    mapper = TravelTimeMap(beta=beta, n_grid=n_grid_map)
    Fx = mapper.forward_F(X)
    tau0 = np.mod(Fx - T, mapper.P)
    s = mapper.inverse_F(tau0)
    return periodic_gaussian_np(s, x0, nu)

# ============================================================
# 4) Predictor: same generic dynamic-basis meta-SPINN
# ============================================================

class MetaSPINNVariableAdvectionDynamic(nn.Module):
    def __init__(self, M=64, hidden=128, n_freq_t=4):
        super().__init__()
        self.M = M
        self.n_freq_t = n_freq_t

        self.base_centers = nn.Parameter(torch.linspace(0.0, 1.0, M + 1)[:-1].clone())
        self.base_log_widths = nn.Parameter(torch.full((M,), -2.2))
        self.base_amp_bias = nn.Parameter(torch.zeros(M))

        in_dim = (1 + 2 * n_freq_t) + 3

        self.temporal_net = nn.Sequential(
            nn.Linear(in_dim, hidden),
            nn.Tanh(),
            nn.Linear(hidden, hidden),
            nn.Tanh(),
            nn.Linear(hidden, 3 * M)
        )

    @staticmethod
    def normalize_p(p):
        p_mean = p.new_tensor([0.5, 0.075, 0.40])
        p_std = p.new_tensor([0.3, 0.03, 0.15])
        return (p - p_mean) / p_std

    def dynamic_params(self, p, t):
        B, N, _ = t.shape
        p_norm = self.normalize_p(p)
        p_rep = p_norm.unsqueeze(1).expand(B, N, 3)
        tf = fourier_time_features(t, n_freq=self.n_freq_t)
        inp = torch.cat([tf, p_rep], dim=-1)

        out = self.temporal_net(inp.reshape(B * N, -1)).reshape(B, N, 3, self.M)
        amp_raw = out[:, :, 0, :]
        shift_raw = out[:, :, 1, :]
        width_raw = out[:, :, 2, :]

        amps = 1.5 * torch.tanh(amp_raw + self.base_amp_bias.view(1, 1, self.M))
        centers = torch.remainder(
            self.base_centers.view(1, 1, self.M) + 0.35 * torch.tanh(shift_raw),
            1.0
        )
        widths = torch.nn.functional.softplus(
            self.base_log_widths.view(1, 1, self.M) + 0.5 * torch.tanh(width_raw)
        ) + 1e-3

        return amps, centers, widths

    def forward(self, p, xt):
        x = xt[:, :, 0:1]
        t = xt[:, :, 1:2]

        amps, centers, widths = self.dynamic_params(p, t)

        d = periodic_distance_torch(x, centers)
        phi = torch.exp(-(d**2) / (2.0 * widths**2))
        v = torch.sum(amps * phi, dim=-1)

        x0 = p[:, 0:1].unsqueeze(1)
        nu = p[:, 1:2].unsqueeze(1)
        u0 = periodic_gaussian(x, x0, nu).squeeze(-1)
        u = u0 + t.squeeze(-1) * v

        return u, (amps, centers, widths)

# ============================================================
# 5) Predictor training
# ============================================================

def sample_p_batch(B, device, x0_range=(0.2, 0.8), nu_range=(0.03, 0.12), beta_range=(0.20, 0.60)):
    x0 = x0_range[0] + (x0_range[1] - x0_range[0]) * torch.rand(B, 1, device=device)
    r = torch.rand(B, 1, device=device)
    log_nu_min = math.log10(nu_range[0])
    log_nu_max = math.log10(nu_range[1])
    nu = 10 ** (log_nu_min + (log_nu_max - log_nu_min) * r)
    beta = beta_range[0] + (beta_range[1] - beta_range[0]) * torch.rand(B, 1, device=device)
    return torch.cat([x0, nu, beta], dim=1)

def sample_uniform_interior(B, N, device):
    x = torch.rand(B, N, 1, device=device)
    t = torch.rand(B, N, 1, device=device)
    return torch.cat([x, t], dim=-1)

def sample_ic_points(B, N, device):
    x = torch.rand(B, N, 1, device=device)
    t = torch.zeros(B, N, 1, device=device)
    return torch.cat([x, t], dim=-1)

def sample_periodic_bc_points(B, N, device):
    t = torch.rand(B, N, 1, device=device)
    xL = torch.zeros(B, N, 1, device=device)
    xR = torch.ones(B, N, 1, device=device)
    xtL = torch.cat([xL, t], dim=-1)
    xtR = torch.cat([xR, t], dim=-1)
    return xtL, xtR

def sample_localized_interior(B, N, p, device):
    x0 = p[:, 0:1]
    nu = p[:, 1:2]
    beta = p[:, 2:3]

    t = torch.rand(B, N, 1, device=device)
    a0 = a_of_x_torch(x0, beta)
    x_center = torch.remainder(x0.unsqueeze(1) + a0.unsqueeze(1) * t, 1.0)

    eps = torch.randn(B, N, 1, device=device)
    width = 2.8 * nu.unsqueeze(1)
    x = torch.remainder(x_center + width * eps, 1.0)
    return torch.cat([x, t], dim=-1)

def advection_residual_batch(model, p, xt_int):
    xt_req = xt_int.clone().detach().requires_grad_(True)
    u, _ = model(p, xt_req)

    grad_u = torch.autograd.grad(
        outputs=u,
        inputs=xt_req,
        grad_outputs=torch.ones_like(u),
        create_graph=True,
        retain_graph=True
    )[0]

    u_x = grad_u[:, :, 0]
    u_t = grad_u[:, :, 1]

    x = xt_req[:, :, 0]
    beta = p[:, 2:3].expand_as(x)
    a = a_of_x_torch(x, beta)
    return u_t + a * u_x

def train_predictor(
    device,
    epochs=5000,
    lr=5e-4,
    M=64,
    hidden=128,
    B_pde=8,
    N_int_uniform=320,
    N_int_local=192,
    N_ic=256,
    N_per=256,
    w_pde=1.0,
    w_ic=1.0,
    w_per=1.0,
    w_sparse=1e-4,
    w_width=1e-6,
    w_motion=1e-6,
):
    model = MetaSPINNVariableAdvectionDynamic(M=M, hidden=hidden, n_freq_t=4).to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr)

    best_loss = float('inf')
    best_state = None

    history = {'epoch': [], 'loss': [], 'loss_pde': [], 'loss_ic': [], 'loss_per': []}

    x0_range = (0.2, 0.8)
    nu_train_range = (0.03, 0.12)
    beta_range = (0.20, 0.60)
    nu_easy = 0.06

    for ep in range(1, epochs + 1):
        model.train()
        optimizer.zero_grad()

        if ep <= epochs // 2:
            nu_range_curr = (nu_easy, nu_train_range[1])
        else:
            nu_range_curr = nu_train_range

        p_batch = sample_p_batch(B_pde, device, x0_range=x0_range, nu_range=nu_range_curr, beta_range=beta_range)
        xt_uni = sample_uniform_interior(B_pde, N_int_uniform, device)
        xt_loc = sample_localized_interior(B_pde, N_int_local, p_batch, device)
        xt_int = torch.cat([xt_uni, xt_loc], dim=1)

        xt_ic = sample_ic_points(B_pde, N_ic, device)
        xtL, xtR = sample_periodic_bc_points(B_pde, N_per, device)

        loss_pde = torch.mean(advection_residual_batch(model, p_batch, xt_int)**2)

        u_ic, _ = model(p_batch, xt_ic)
        x_ic = xt_ic[:, :, 0]
        x0 = p_batch[:, 0:1].expand_as(x_ic)
        nu = p_batch[:, 1:2].expand_as(x_ic)
        u_ic_true = periodic_gaussian(x_ic, x0, nu)
        loss_ic = torch.mean((u_ic - u_ic_true)**2)

        uL, _ = model(p_batch, xtL)
        uR, _ = model(p_batch, xtR)
        loss_per = torch.mean((uL - uR)**2)

        t_reg = torch.rand(B_pde, 64, 1, device=device)
        amps, centers, widths = model.dynamic_params(p_batch, t_reg)
        loss_sparse = w_sparse * torch.mean(torch.abs(amps))
        loss_width = w_width * torch.mean((1.0 / (widths + 1e-6))**2)

        t1 = torch.rand(B_pde, 64, 1, device=device)
        t2 = torch.clamp(t1 + 0.05, 0.0, 1.0)
        _, centers1, _ = model.dynamic_params(p_batch, t1)
        _, centers2, _ = model.dynamic_params(p_batch, t2)
        center_jump = periodic_distance_torch(centers2, centers1)
        loss_motion = w_motion * torch.mean(center_jump**2)

        loss = w_pde * loss_pde + w_ic * loss_ic + w_per * loss_per + loss_sparse + loss_width + loss_motion
        loss.backward()
        optimizer.step()

        history['epoch'].append(ep)
        history['loss'].append(loss.item())
        history['loss_pde'].append(loss_pde.item())
        history['loss_ic'].append(loss_ic.item())
        history['loss_per'].append(loss_per.item())

        if loss.item() < best_loss:
            best_loss = loss.item()
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

        if ep % 200 == 0 or ep == 1:
            print(
                f"Predictor Epoch {ep:4d} | Loss {loss.item():.3e} | "
                f"PDE {loss_pde.item():.3e} | IC {loss_ic.item():.3e} | PER {loss_per.item():.3e}"
            )

    if best_state is not None:
        model.load_state_dict({k: v.to(device) for k, v in best_state.items()})

    return model, history

# ============================================================
# 6) Refined frozen hidden geometry for PIELM
# ============================================================

def build_fixed_background_geometry(n_fix_x=16, n_fix_t=12, sigma_x_factor=1.15, sigma_t_factor=1.15):
    """
    Denser fixed background centers.
    Widths are tied to grid spacing so that increasing the number of fixed
    centers automatically reduces their widths.
    """
    x_fix = np.linspace(0.0, 1.0, n_fix_x, endpoint=False)
    t_fix = np.linspace(0.0, 1.0, n_fix_t)
    XX, TT = np.meshgrid(x_fix, t_fix, indexing='ij')

    dx = 1.0 / float(n_fix_x)
    dt = 1.0 / float(max(n_fix_t - 1, 1))

    fix_mu_x = XX.ravel()
    fix_mu_t = TT.ravel()
    fix_sigma_x = (sigma_x_factor * dx) * np.ones_like(fix_mu_x)
    fix_sigma_t = (sigma_t_factor * dt) * np.ones_like(fix_mu_t)

    return fix_mu_x, fix_mu_t, fix_sigma_x, fix_sigma_t

def extract_dynamic_centers_from_predictor(
    predictor,
    p_test,
    device,
    n_time_guides=20,
    n_dyn_keep=12,
    sigma_t_scale=1.2,
):
    """
    Existing dynamic centers:
    choose strongest predictor internal centers by |amplitude| at each guide time.
    """
    p_t = torch.tensor([p_test], dtype=torch.float32, device=device)
    guide_times = torch.linspace(0.0, 1.0, n_time_guides, device=device).view(1, n_time_guides, 1)

    with torch.no_grad():
        amps, centers, widths = predictor.dynamic_params(p_t, guide_times)

    amps = amps[0].cpu().numpy()
    centers = centers[0].cpu().numpy()
    widths = widths[0].cpu().numpy()
    times_np = guide_times[0, :, 0].cpu().numpy()
    dt = 1.0 / max(n_time_guides - 1, 1)

    mu_x, mu_t, sigma_x, sigma_t = [], [], [], []
    for k, tval in enumerate(times_np):
        score = np.abs(amps[k])
        idx = np.argsort(score)[::-1][:n_dyn_keep]
        for j in idx:
            mu_x.append(float(centers[k, j]))
            mu_t.append(float(tval))
            sigma_x.append(float(widths[k, j]))
            sigma_t.append(float(sigma_t_scale * dt))

    return (np.asarray(mu_x), np.asarray(mu_t),
            np.asarray(sigma_x), np.asarray(sigma_t))

def extract_gradient_refinement_centers(
    predictor,
    p_test,
    device,
    n_time_guides=20,
    n_grad_keep=12,
    grad_grid_x=320,
    sigma_x_scale=0.45,
    sigma_t_scale=0.70,
):
    """
    New refinement centers:
    - evaluate predictor gradient |u_x| on a fine x-grid for each guide time
    - pick strongest locations
    - assign widths smaller than the fixed-background widths
    """
    x0_test, nu_test, beta_test = [float(v) for v in p_test]
    x_grid = np.linspace(0.0, 1.0, grad_grid_x, endpoint=False)
    guide_times = np.linspace(0.0, 1.0, n_time_guides)

    dx = 1.0 / float(grad_grid_x)
    dt = 1.0 / float(max(n_time_guides - 1, 1))

    mu_x, mu_t, sigma_x, sigma_t = [], [], [], []

    for tval in guide_times:
        xt = np.stack([x_grid, tval * np.ones_like(x_grid)], axis=1)[None, :, :]
        xt_t = torch.tensor(xt, dtype=torch.float32, device=device, requires_grad=True)
        p_t = torch.tensor([[x0_test, nu_test, beta_test]], dtype=torch.float32, device=device)

        u_pred, _ = predictor(p_t, xt_t)
        grads = torch.autograd.grad(
            outputs=u_pred,
            inputs=xt_t,
            grad_outputs=torch.ones_like(u_pred),
            create_graph=False,
            retain_graph=False
        )[0]

        ux = grads[0, :, 0].detach().cpu().numpy()
        g = np.abs(ux)

        idx = np.argsort(g)[::-1]
        picked = []
        for j in idx:
            xj = x_grid[j]
            if all(abs(periodic_distance_np(xj, xp)) > 4.0 * dx for xp in picked):
                picked.append(xj)
            if len(picked) >= n_grad_keep:
                break

        for xj in picked:
            mu_x.append(float(xj))
            mu_t.append(float(tval))
            sigma_x.append(float(max(1.5 * dx, sigma_x_scale * nu_test)))
            sigma_t.append(float(max(0.35 * dt, sigma_t_scale * dt)))

    return (np.asarray(mu_x), np.asarray(mu_t),
            np.asarray(sigma_x), np.asarray(sigma_t))

def build_refined_frozen_hidden_geometry(
    predictor,
    p_test,
    device,
    n_time_guides=20,
    n_dyn_keep=12,
    n_grad_keep=12,
    n_fix_x=16,
    n_fix_t=12,
    sigma_t_scale_dyn=1.2,
):
    dyn_mu_x, dyn_mu_t, dyn_sigma_x, dyn_sigma_t = extract_dynamic_centers_from_predictor(
        predictor=predictor,
        p_test=p_test,
        device=device,
        n_time_guides=n_time_guides,
        n_dyn_keep=n_dyn_keep,
        sigma_t_scale=sigma_t_scale_dyn,
    )

    grad_mu_x, grad_mu_t, grad_sigma_x, grad_sigma_t = extract_gradient_refinement_centers(
        predictor=predictor,
        p_test=p_test,
        device=device,
        n_time_guides=n_time_guides,
        n_grad_keep=n_grad_keep,
        grad_grid_x=320,
        sigma_x_scale=0.45,
        sigma_t_scale=0.70,
    )

    fix_mu_x, fix_mu_t, fix_sigma_x, fix_sigma_t = build_fixed_background_geometry(
        n_fix_x=n_fix_x,
        n_fix_t=n_fix_t,
        sigma_x_factor=1.15,
        sigma_t_factor=1.15,
    )

    mu_x = np.concatenate([dyn_mu_x, grad_mu_x, fix_mu_x], axis=0)
    mu_t = np.concatenate([dyn_mu_t, grad_mu_t, fix_mu_t], axis=0)
    sigma_x = np.concatenate([dyn_sigma_x, grad_sigma_x, fix_sigma_x], axis=0)
    sigma_t = np.concatenate([dyn_sigma_t, grad_sigma_t, fix_sigma_t], axis=0)

    return {
        'mu_x': mu_x,
        'mu_t': mu_t,
        'sigma_x': sigma_x,
        'sigma_t': sigma_t,
    }

# ============================================================
# 7) PIELM corrector
# ============================================================

def eval_spacetime_rbf_np(x, t, mu_x, mu_t, sigma_x, sigma_t):
    x = np.asarray(x).reshape(-1, 1)
    t = np.asarray(t).reshape(-1, 1)

    dx = periodic_distance_np(x, mu_x.reshape(1, -1))
    dt = t - mu_t.reshape(1, -1)

    Phi = np.exp(
        -(dx**2) / (2.0 * sigma_x.reshape(1, -1)**2)
        -(dt**2) / (2.0 * sigma_t.reshape(1, -1)**2)
    )
    return Phi

def eval_spacetime_rbf_derivatives_np(x, t, mu_x, mu_t, sigma_x, sigma_t):
    x = np.asarray(x).reshape(-1, 1)
    t = np.asarray(t).reshape(-1, 1)

    dx = periodic_distance_np(x, mu_x.reshape(1, -1))
    dt = t - mu_t.reshape(1, -1)

    Phi = np.exp(
        -(dx**2) / (2.0 * sigma_x.reshape(1, -1)**2)
        -(dt**2) / (2.0 * sigma_t.reshape(1, -1)**2)
    )

    Phi_x = -(dx / (sigma_x.reshape(1, -1)**2)) * Phi
    Phi_t = -(dt / (sigma_t.reshape(1, -1)**2)) * Phi
    return Phi, Phi_x, Phi_t

def solve_pielm_corrector_for_task(
    p_test,
    hidden_geom,
    N_res=7000,
    N_bc=1600,
    lambda_ridge=1e-8,
    seed=1234,
):
    x0_test, nu_test, beta_test = [float(v) for v in p_test]
    mu_x = hidden_geom['mu_x']
    mu_t = hidden_geom['mu_t']
    sigma_x = hidden_geom['sigma_x']
    sigma_t = hidden_geom['sigma_t']
    M = len(mu_x)

    rng = np.random.default_rng(seed)

    x_res = rng.random(N_res)
    t_res = rng.random(N_res)

    Phi, Phi_x, Phi_t = eval_spacetime_rbf_derivatives_np(x_res, t_res, mu_x, mu_t, sigma_x, sigma_t)

    a_res = a_of_x_np(x_res, beta_test)
    u0_x_res = -(periodic_distance_np(x_res, x0_test) / (nu_test**2)) * periodic_gaussian_np(x_res, x0_test, nu_test)

    A_res = Phi + t_res[:, None] * Phi_t + (a_res[:, None] * t_res[:, None]) * Phi_x
    b_res = -a_res * u0_x_res

    t_bc = rng.random(N_bc)
    xL = np.zeros(N_bc)
    xR = np.ones(N_bc)

    PhiL = eval_spacetime_rbf_np(xL, t_bc, mu_x, mu_t, sigma_x, sigma_t)
    PhiR = eval_spacetime_rbf_np(xR, t_bc, mu_x, mu_t, sigma_x, sigma_t)

    A_bc = t_bc[:, None] * (PhiL - PhiR)
    b_bc = np.zeros(N_bc)

    _, PhiL_x, _ = eval_spacetime_rbf_derivatives_np(xL, t_bc, mu_x, mu_t, sigma_x, sigma_t)
    _, PhiR_x, _ = eval_spacetime_rbf_derivatives_np(xR, t_bc, mu_x, mu_t, sigma_x, sigma_t)

    A_db = t_bc[:, None] * (PhiL_x - PhiR_x)
    b_db = np.zeros(N_bc)

    A = np.vstack([A_res, A_bc, A_db])
    b = np.concatenate([b_res, b_bc, b_db])

    ATA = A.T @ A
    ATb = A.T @ b
    coeffs = np.linalg.solve(ATA + lambda_ridge * np.eye(M), ATb)

    return coeffs, {'A_shape': A.shape, 'M': M, 'lambda_ridge': lambda_ridge}

def evaluate_pielm_solution_on_grid(p_test, hidden_geom, coeffs, Nx=220, Nt=220, map_grid=20001):
    x0_test, nu_test, beta_test = [float(v) for v in p_test]

    x = np.linspace(0.0, 1.0, Nx, endpoint=False)
    t = np.linspace(0.0, 1.0, Nt)
    X, T = np.meshgrid(x, t, indexing='ij')

    Phi = eval_spacetime_rbf_np(
        X.ravel(), T.ravel(),
        hidden_geom['mu_x'], hidden_geom['mu_t'],
        hidden_geom['sigma_x'], hidden_geom['sigma_t']
    )

    correction = (T.ravel()[:, None] * Phi) @ coeffs
    u0 = periodic_gaussian_np(X.ravel(), x0_test, nu_test)
    u_corr = (u0 + correction).reshape(Nx, Nt)

    u_exact = exact_solution_variable_speed(X, T, x0_test, nu_test, beta_test, n_grid_map=map_grid)
    abs_err = np.abs(u_corr - u_exact)
    rel_l2 = np.linalg.norm((u_corr - u_exact).ravel()) / (np.linalg.norm(u_exact.ravel()) + 1e-14)

    return {'u_corr': u_corr, 'u_exact': u_exact, 'abs_err': abs_err, 'rel_l2': rel_l2}

# ============================================================
# 8) Predictor evaluation
# ============================================================

def evaluate_predictor_on_grid(model, p_test, device, Nx=220, Nt=220, map_grid=20001):
    x0_test, nu_test, beta_test = [float(v) for v in p_test]

    x = np.linspace(0.0, 1.0, Nx, endpoint=False)
    t = np.linspace(0.0, 1.0, Nt)
    X, T = np.meshgrid(x, t, indexing='ij')

    xt_grid = np.stack([X.ravel(), T.ravel()], axis=1)
    xt_grid_t = torch.tensor(xt_grid, dtype=torch.float32, device=device).view(1, -1, 2)
    p_t = torch.tensor([p_test], dtype=torch.float32, device=device)

    model.eval()
    with torch.no_grad():
        u_pred_flat, _ = model(p_t, xt_grid_t)

    u_pred = u_pred_flat.cpu().numpy().reshape(Nx, Nt)
    u_exact = exact_solution_variable_speed(X, T, x0_test, nu_test, beta_test, n_grid_map=map_grid)
    abs_err = np.abs(u_pred - u_exact)
    rel_l2 = np.linalg.norm((u_pred - u_exact).ravel()) / (np.linalg.norm(u_exact.ravel()) + 1e-14)

    return {'u_pred': u_pred, 'u_exact': u_exact, 'abs_err': abs_err, 'rel_l2': rel_l2}

# ============================================================
# 9) Plotting
# ============================================================

def plot_training_history(history, save_path):
    plt.figure(figsize=(7, 4.5))
    plt.plot(history['epoch'], history['loss'], lw=2, label='Total')
    plt.plot(history['epoch'], history['loss_pde'], lw=1.5, label='PDE')
    plt.plot(history['epoch'], history['loss_ic'], lw=1.5, label='IC')
    plt.plot(history['epoch'], history['loss_per'], lw=1.5, label='Periodic BC')
    plt.yscale('log')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.title('Predictor training loss history')
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.savefig(save_path, dpi=300)
    plt.close()

def plot_multicase_corrector_results(
    predictor,
    test_cases,
    device,
    save_dir,
    n_time_guides=20,
    n_dyn_keep=12,
    n_grad_keep=12,
    n_fix_x=16,
    n_fix_t=12,
    lambda_ridge=1e-8,
    Nx=220,
    Nt=220,
):
    num_tests = len(test_cases)

    # Standard template geometry: same as FIG_TC_03_A_SOLUTION
    fig, axs = plt.subplots(
        4, num_tests,
        figsize=(5.4 * num_tests, 17.6),
        constrained_layout=False
    )
    if num_tests == 1:
        axs = axs.reshape(4, 1)

    # Standard template margins/spacings
    fig.subplots_adjust(
        left=0.08,
        right=0.93,
        bottom=0.12,
        top=0.98,
        wspace=0.08,
        hspace=0.22
    )

    summary = []
    exact_list = []
    pred_list = []
    corr_list = []
    err_list = []

    for name, x0_test, nu_test, beta_test in test_cases:
        p_test = (x0_test, nu_test, beta_test)

        pred_result = evaluate_predictor_on_grid(
            predictor, p_test, device, Nx=Nx, Nt=Nt
        )

        hidden_geom = build_refined_frozen_hidden_geometry(
            predictor=predictor,
            p_test=p_test,
            device=device,
            n_time_guides=n_time_guides,
            n_dyn_keep=n_dyn_keep,
            n_grad_keep=n_grad_keep,
            n_fix_x=n_fix_x,
            n_fix_t=n_fix_t,
            sigma_t_scale_dyn=1.2,
        )

        coeffs, info = solve_pielm_corrector_for_task(
            p_test=p_test,
            hidden_geom=hidden_geom,
            N_res=7000,
            N_bc=1600,
            lambda_ridge=lambda_ridge,
            seed=SEED,
        )

        corr_result = evaluate_pielm_solution_on_grid(
            p_test=p_test,
            hidden_geom=hidden_geom,
            coeffs=coeffs,
            Nx=Nx,
            Nt=Nt,
        )

        exact_list.append((x0_test, nu_test, beta_test, corr_result["u_exact"]))
        pred_list.append((x0_test, nu_test, beta_test, pred_result["u_pred"]))
        corr_list.append((x0_test, nu_test, beta_test, corr_result["u_corr"]))
        err_list.append((x0_test, nu_test, beta_test, corr_result["abs_err"]))

        summary.append({
            "name": name,
            "x0": x0_test,
            "nu": nu_test,
            "beta": beta_test,
            "predictor_rel_l2": pred_result["rel_l2"],
            "corrector_rel_l2": corr_result["rel_l2"],
            "n_hidden": int(info["M"]),
        })

    # Shared color scales across all panels
    all_sol_vals = np.concatenate(
        [u.ravel() for _, _, _, u in exact_list + pred_list + corr_list]
    )
    sol_vmin = float(np.min(all_sol_vals))
    sol_vmax = float(np.max(all_sol_vals))

    all_err_vals = np.concatenate([u.ravel() for _, _, _, u in err_list])
    err_vmin = 0.0
    err_vmax = float(np.max(all_err_vals))

    sol_images = []
    err_images = []

    for j in range(num_tests):
        x0_test, nu_test, beta_test, u_exact = exact_list[j]
        _, _, _, u_pred = pred_list[j]
        _, _, _, u_corr = corr_list[j]
        _, _, _, u_err = err_list[j]

        # ----- Row 1: exact -----
        ax0 = axs[0, j]
        im0 = ax0.imshow(
            u_exact.T,
            origin="lower",
            extent=[0, 1, 0, 1],
            aspect="auto",
            cmap="viridis",
            vmin=sol_vmin,
            vmax=sol_vmax,
        )
        sol_images.append(im0)
        ax0.set_title(
            rf"$(x_0,\nu,\beta)=({x0_test:.2f},{nu_test:.2f},{beta_test:.2f})$",
            fontsize=20,
            pad=8
        )

        # ----- Row 2: predictor -----
        ax1 = axs[1, j]
        ax1.imshow(
            u_pred.T,
            origin="lower",
            extent=[0, 1, 0, 1],
            aspect="auto",
            cmap="viridis",
            vmin=sol_vmin,
            vmax=sol_vmax,
        )

        # ----- Row 3: corrector -----
        ax2 = axs[2, j]
        ax2.imshow(
            u_corr.T,
            origin="lower",
            extent=[0, 1, 0, 1],
            aspect="auto",
            cmap="viridis",
            vmin=sol_vmin,
            vmax=sol_vmax,
        )

        # ----- Row 4: absolute error -----
        ax3 = axs[3, j]
        im3 = ax3.imshow(
            u_err.T,
            origin="lower",
            extent=[0, 1, 0, 1],
            aspect="auto",
            cmap="magma",
            vmin=err_vmin,
            vmax=err_vmax,
        )
        err_images.append(im3)

        # Standard axis formatting
        for ax in [ax0, ax1, ax2, ax3]:
            ax.set_xlabel("x", fontsize=18)
            ax.tick_params(axis="both", labelsize=12)
            ax.set_xticks([0.0, 0.5, 1.0])

        # Standard y-axis behavior
        for ax in [ax0, ax1, ax2, ax3]:
            if j == 0:
                ax.set_ylabel("t", fontsize=18, labelpad=8)
                ax.set_yticks([0.0, 0.5, 1.0])
            else:
                ax.set_ylabel("")
                ax.set_yticks([])

    # Standard row labels
    axs[0, 0].text(
        -0.30, 0.5, r"$u_{\mathrm{exact}}$",
        transform=axs[0, 0].transAxes,
        rotation=90, va="center", ha="center", fontsize=28
    )
    axs[1, 0].text(
        -0.30, 0.5, r"$u_{\mathrm{pred}}$",
        transform=axs[1, 0].transAxes,
        rotation=90, va="center", ha="center", fontsize=28
    )
    axs[2, 0].text(
        -0.30, 0.5, r"$u_{\mathrm{corr}}$",
        transform=axs[2, 0].transAxes,
        rotation=90, va="center", ha="center", fontsize=28
    )
    axs[3, 0].text(
        -0.30, 0.5, r"$|u_{\mathrm{corr}}-u_{\mathrm{exact}}|$",
        transform=axs[3, 0].transAxes,
        rotation=90, va="center", ha="center", fontsize=28
    )

    # Standard colorbars
    cbar1 = fig.colorbar(
        sol_images[0],
        ax=axs[0:3, :],
        fraction=0.025,
        pad=0.015,
        shrink=0.95
    )
    cbar1.set_label(r"$u(x,t)$", fontsize=24)
    cbar1.ax.tick_params(labelsize=20)

    cbar2 = fig.colorbar(
        err_images[0],
        ax=axs[3, :],
        fraction=0.025,
        pad=0.015,
        shrink=0.92
    )
    cbar2.set_label("Absolute Error", fontsize=24)
    cbar2.ax.tick_params(labelsize=20)

    # Standard footer placement
    fig.text(
        0.5, 0.035,
        r"Training range: $x_0 \in [0.20,0.80],\ \nu \in [0.03,0.12],\ \beta \in [0.20,0.60]$",
        ha="center", va="center", fontsize=24
    )

    png_path = os.path.join(save_dir, "FIG_TC_04_A_SOLUTION.png")
    pdf_path = os.path.join(save_dir, "FIG_TC_04_A_SOLUTION.pdf")
    fig.savefig(png_path, dpi=300, bbox_inches="tight")
    fig.savefig(pdf_path, dpi=300, bbox_inches="tight")
    plt.close(fig)

    return summary, png_path
def save_summary_csv(summary, save_dir):
    csv_path = os.path.join(save_dir, 'predictor_corrector_refined_summary.csv')
    fields = ['name', 'x0', 'nu', 'beta', 'predictor_rel_l2', 'corrector_rel_l2', 'n_hidden']
    with open(csv_path, 'w', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=fields)
        writer.writeheader()
        for row in summary:
            writer.writerow(row)
    return csv_path

# ============================================================
# 10) Main
# ============================================================


# ============================================================
# 10) Main: load trained model and run post-processing only
# ============================================================

def load_history_npz(history_path):
    data = np.load(history_path)
    history = {
        'epoch': data['epoch'].tolist(),
        'loss': data['loss'].tolist(),
        'loss_pde': data['loss_pde'].tolist(),
        'loss_ic': data['loss_ic'].tolist(),
        'loss_per': data['loss_per'].tolist(),
    }
    return history

def main():
    parser = argparse.ArgumentParser(description='Variable-speed advection predictor-corrector post-processing only')
    parser.add_argument('--gpu', action='store_true', help='Prefer CUDA if available')
    parser.add_argument('--M', type=int, default=64, help='Predictor basis count')
    parser.add_argument('--hidden', type=int, default=128)
    parser.add_argument('--outdir', type=str, default='TC_04_FIGURES')
    parser.add_argument('--model_path', type=str, default='predictor_model.pt')
    parser.add_argument('--history_path', type=str, default='predictor_training_history.npz')
    parser.add_argument('--n_time_guides', type=int, default=20)
    parser.add_argument('--n_dyn_keep', type=int, default=12)
    parser.add_argument('--n_grad_keep', type=int, default=12)
    parser.add_argument('--n_fix_x', type=int, default=16)
    parser.add_argument('--n_fix_t', type=int, default=12)
    parser.add_argument('--lambda_ridge', type=float, default=1e-8)
    parser.add_argument('--Nx', type=int, default=220)
    parser.add_argument('--Nt', type=int, default=220)
    args, _unknown = parser.parse_known_args()

    device = torch.device('cuda' if (args.gpu and torch.cuda.is_available()) else ('cuda' if torch.cuda.is_available() else 'cpu'))
    print('Using device:', device)

    os.makedirs(args.outdir, exist_ok=True)
    model_path = os.path.join(args.outdir, args.model_path)

    loaded_predictor = MetaSPINNVariableAdvectionDynamic(M=args.M, hidden=args.hidden, n_freq_t=4).to(device)
    loaded_predictor.load_state_dict(torch.load(model_path, map_location=device))
    loaded_predictor.eval()
    print(f'Loaded predictor from: {model_path}')

    history_path = os.path.join(args.outdir, args.history_path)
    if os.path.exists(history_path):
        history = load_history_npz(history_path)
        print(f'Loaded training history from: {history_path}')
        train_plot = os.path.join(args.outdir, 'predictor_training_loss_replot.png')
        plot_training_history(history, train_plot)
        print(f'Saved: {train_plot}')

    test_cases = [
        ('center_in_range',       0.50, 0.07, 0.35),
        ('off_center_in_range',   0.30, 0.09, 0.55),
        ('narrow_in_range',       0.65, 0.04, 0.45),
        ('narrow_beta_out_range', 0.50, 0.04, 0.70),
    ]

    summary, fig_path = plot_multicase_corrector_results(
        predictor=loaded_predictor,
        test_cases=test_cases,
        device=device,
        save_dir=args.outdir,
        n_time_guides=args.n_time_guides,
        n_dyn_keep=args.n_dyn_keep,
        n_grad_keep=args.n_grad_keep,
        n_fix_x=args.n_fix_x,
        n_fix_t=args.n_fix_t,
        lambda_ridge=args.lambda_ridge,
        Nx=args.Nx,
        Nt=args.Nt,
    )
    print(f'Saved: {fig_path}')

    csv_path = save_summary_csv(summary, args.outdir)
    print(f'Saved: {csv_path}')

    print('\nSummary of test cases:')
    for row in summary:
        print(
            f"{row['name']:22s} | x0={row['x0']:.3f}, nu={row['nu']:.3f}, beta={row['beta']:.3f} | "
            f"predictor relL2={row['predictor_rel_l2']:.3e} | "
            f"corrector relL2={row['corrector_rel_l2']:.3e} | "
            f"n_hidden={row['n_hidden']}"
        )

    print(f'\nAll outputs saved in: {args.outdir}')

if __name__ == '__main__':
    main()

Using device: cpu
Loaded predictor from: TC_04_FIGURES/predictor_model.pt
Loaded training history from: TC_04_FIGURES/predictor_training_history.npz


/tmp/ipykernel_22554/142691769.py:998: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  loaded_predictor.load_state_dict(torch.load(model_path, map_location=device))


Saved: TC_04_FIGURES/predictor_training_loss_replot.png
Saved: TC_04_FIGURES/FIG_TC_04_A_SOLUTION.png
Saved: TC_04_FIGURES/predictor_corrector_refined_summary.csv

Summary of test cases:
center_in_range        | x0=0.500, nu=0.070, beta=0.350 | predictor relL2=4.111e-01 | corrector relL2=2.673e-02 | n_hidden=672
off_center_in_range    | x0=0.300, nu=0.090, beta=0.550 | predictor relL2=5.614e-01 | corrector relL2=4.484e-02 | n_hidden=672
narrow_in_range        | x0=0.650, nu=0.040, beta=0.450 | predictor relL2=6.249e-01 | corrector relL2=5.715e-02 | n_hidden=672
narrow_beta_out_range  | x0=0.500, nu=0.040, beta=0.700 | predictor relL2=1.075e+00 | corrector relL2=3.896e-01 | n_hidden=672

All outputs saved in: TC_04_FIGURES
